## Imaging Star Clusters Tutorial:

This is a tutorial to teach you how to make images of star clusters using data from FIRE 2 Simulations.

### Initial set up
Make sure you have downloaded all files from the github link: https://github.com/Amoac/star-cluster-imaging 

There is a variety of imports to make the code work so you may need to use pip install to successfully run the code. You will need to edit the sys.path.append to the location of your 'simulations_code' file and edit the "kernel_file_path" to the location of your "kernel2d" file.

Additional information about the functions in this code can be found in star_cluster_imaging.py

In [ ]:
import sys
sys.path.append("/home1/09528/amoac/simulations_code/") 
import gizmo_analysis as gizmo 
import utilities as ut
from fof_analysis import fof
from fof_analysis import star_cluster_imaging
import numpy as np
import pandas as pd
from matplotlib import pylab as plt
from matplotlib import rc #to use Latex math symbols like 'phi'
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec
import seaborn as sns
from astropy.table import Table
from astropy.io import ascii
import pickle
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.colorbar as colorbar
from importlib import reload
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

#simulation files path
simname     = 'm12i_res7100'
simdir      = '/scratch/07428/binod/GalaxiesOnFIRE/cr_suite/m12i_r7100/mhdcv/1Myr/fire2/'
kernel_file_path = '/home1/09528/amoac/'

If you make edits to any of the functions in the star_cluster_imaging package you will need to reload them with the function below:

In [ ]:
reload(star_cluster_imaging) 

### Getting Started
We will start set up by establishing some units to be used in future calculations and initializing dictionaries that will be used to contain the gas and star properties that we will use to produce images.

In [ ]:
#establishing some units 
b_parsec    = 4
b_kpc       = b_parsec / 1000.
ncut_min    = 5
softening   = 4. #res 7100

# Initialize dictionaries to store properties
gas_star_properties_data = {}
part = {}

***Choosing Snapshots:*** you can choose any snapshot you like ranging from 576 to 696. 
One snapshot takes ~3 minutes to load.

In [ ]:
snaps = [694] #set snaps to any value from 576 to 696

In [ ]:
# This loop automatically stores snapshot data for any number of snapshots (this will be important for imaging multiple snapshots at once)
for snapshot in snaps:
    # Read the snapshot data for the current snapshot
    part[snapshot] = gizmo.io.Read.read_snapshots(['star', 'gas'],'index', snapshot, simulation_directory=simdir, assign_hosts_rotation=True,assign_hosts=True)
    # Get star properties for the current snapshot
    gas_star_properties_data[snapshot] = star_cluster_imaging.get_gas_star_properties(part[snapshot])

***Choosing your star cluster***: You can select which star cluster you would like to view after running the cell below once.

You can change the filter for age ranges of the stars by editing the value of age_cut_Myr_Max and age_cut_Min_Myr. This can reduce or increase the number of stars that will be imaged later on.

In [ ]:
# Process the snapshots to get initial cluster information
cluster_select = 0 #change this to any cluster as long as it is within range of the 'number of groups' value after running first_snapshot_data, 0 being the first cluster 
first_snapshot_data = star_cluster_imaging.process_first_snapshot(gas_star_properties_data[snaps[0]],age_cut_Max_Myr=2, age_cut_Min_Myr=1, cluster_select = cluster_select )

***Variables and lists to be initialized***

Editing pc_width changes how zoomed in you are to the cluster. Higher values zoom out more while smaller values zoom in.

In [ ]:
# Initialize variables
pc_width = 40 # this changes the zoom of the image. Based on this value the total width of the picture will be 2*pc_width
kpc_width = pc_width / 1e3
num_snapshots = len(snaps)

# Create a list to store each snapshot's XY plane image data
xy_images = []
titles = []

### Imaging Making Your First Star Cluster Picture

***Editing the size of your plots:***

By changing the value of plot_size, you can make your plots bigger or smaller

In [ ]:
plot_size = 5
fig = plt.figure(figsize=(num_snapshots*plot_size, plot_size))
gs0 = gridspec.GridSpec(1, num_snapshots+1, width_ratios= [1] * num_snapshots + [0.1])

***Below you will learn how to create an image of a star cluster while viewed on the XY plane.***

These lines of code are preparing and organizing the star cluster data for the current snapshot. The goal is to accurately track the stars from the first snapshot across all subsequent snapshots, ensuring that the images generated reflect the proper evolution of the same group of stars over time. The preserve_ids function is particularly crucial as it maintains this continuity, which is essential for making meaningful comparisons between snapshots.

In [ ]:
gs = gas_star_properties_data[snapshot]
   
s_loc = star_cluster_imaging.preserve_ids(
    first_snapshot_data['ids_cluster'],
    first_snapshot_data['id_generation_cluster'],
    first_snapshot_data['id_child_cluster'], 
    gs['ids'], 
    gs['id_generation'],
    gs['id_child'])


This part of the code is responsible for preparing and processing the data related to the star cluster and surrounding gas for the current snapshot. It identifies and organizes the relevant star and gas particles, applies smoothing to the gas data, and prepares the data for visualization. This ensures that the resulting images are accurate, clear, and focus on the key features of the star cluster’s evolution.

In [ ]:
#editing vmin or vmax values will change how the gas appears on the plots
vmin_value = 2.5e2  # Minimum value for the color bar
vmax_value = 5e4    # Maximum value for the color bar

cluster_info = star_cluster_imaging.find_new_snap_info(s_loc, gs)

mass_cluster = cluster_info['mass_cluster']
x_cluster = cluster_info['x_cluster']
y_cluster = cluster_info['y_cluster']
z_cluster = cluster_info['z_cluster']
    

xcm1 = first_snapshot_data['sxcm'][cluster_select]
ycm1 = first_snapshot_data['sycm'][cluster_select]
zcm1 = first_snapshot_data['szcm'][cluster_select]

#creating filter that will only select the young stars that are visible in the picture
g_ind, s_ind = star_cluster_imaging.get_gas_star_indices(xcm1, ycm1, zcm1, gs, pc_width=pc_width)

gas_star_coordinates = star_cluster_imaging.gas_star_coordinates(gs, g_ind, x_cluster, y_cluster, z_cluster)

#Gaussian Smoothing for gas particles
vel_array_xy, res_xy, extent_xy = star_cluster_imaging.compute_velocity_array_xy(xcm1, ycm1, zcm1, gas_star_coordinates, kernel_file_path, pc_width=pc_width, stepn=13) # change the value of stepn to change the gas resolution
vel_array_xy = np.where(vel_array_xy < vmin_value, vmin_value, vel_array_xy) #This line is important for filling in any empty spaces in the gaussian smoothing. without it you end up with ugly white spaces when there is not gas in a region. 

***Display the image***

In [ ]:
ax = plt.subplot(gs0[0])
ax.get_xaxis().set_visible(False) #remove x-axis labels
ax.get_yaxis().set_visible(False) #remove y-axis labels
ax.set_xlim(xcm1 - kpc_width, xcm1 + kpc_width) #keeping the image within kpc_width
ax.set_ylim(ycm1 - kpc_width, ycm1 + kpc_width) #keeping the image within kpc_width

#PLOT GAS 
ax.imshow(vel_array_xy.T / ((res_xy * 1000.) ** 2), interpolation='gaussian', norm=LogNorm(vmin=vmin_value, vmax=vmax_value), extent=extent_xy, origin='lower')

#PLOT STARS 
ax.scatter(x_cluster, y_cluster, s=15, marker='*', edgecolors='black', facecolors='yellow', linewidth=0.5) #This plots the stars, you can edit the colors and sizes to your liking

plt.show() #Display the image

***Final Touches***

We have finally created our picture! Lets add some labels and other helpful indicatiors.

Below demonstrates a few examples for lables you may use including a color bar, realistic times, a scale bar, or just plain text:

In [ ]:
ax = plt.subplot(gs0[0])
ax.get_xaxis().set_visible(False) #remove x-axis labels
ax.get_yaxis().set_visible(False) #remove y-axis labels
ax.set_xlim(xcm1 - kpc_width, xcm1 + kpc_width) #keeping the image within kpc_width
ax.set_ylim(ycm1 - kpc_width, ycm1 + kpc_width) #keeping the image within kpc_width

#PLOT GAS 
ax.imshow(vel_array_xy.T / ((res_xy * 1000.) ** 2), interpolation='gaussian', norm=LogNorm(vmin=vmin_value, vmax=vmax_value), extent=extent_xy, origin='lower')

#PLOT STARS 
ax.scatter(x_cluster, y_cluster, s=15, marker='*', edgecolors='black', facecolors='yellow', linewidth=0.5) #This plots the stars, you can edit the colors and sizes to your liking

### Plain text
ax.text(0.01 ,0.99,'X Y', ha = 'left', va ='top', transform = ax.transAxes, c = 'white')
ax.text(0.99 ,0.01, snaps[0], ha = 'right', va ='bottom', transform = ax.transAxes, c = 'white')

### Making Time Labels
snapshot_times = simdir + '/snapshot_times.txt' # Reading in the snapshot times from a text file to create titles for the images
snapnumber = snaps[0] 

# Reading the snapshot times and associated data from the text file
snaptime_data = np.genfromtxt(snapshot_times, usecols=(0, 3), skip_header=4, dtype=float)
snaps1 = np.array(snaptime_data[:, 0]) # Extracting the snapshot numbers (first column) from the data
times = np.array(snaptime_data[:, 1]) # Extracting the corresponding times (fourth column) from the data
snaptime = times[np.where(snaps1 == snapnumber)][0]# Finding the time associated with the current snapshot number
titles.append(f'XY VIEW {snaptime:.3f} Gyr')# Creating a title string that includes the snapshot time for the XY view
title = f'{snaptime:.3f}' + ' Gyr'# Creating a general title string for the snapshot time, formatted to 3 decimal places
ax.text(0.99 ,0.99,title, ha = 'right', va ='top', transform = ax.transAxes, c = 'white') #positioning and adding time label 


### Scalebar
scalebar_length = 20 #pc    
scalebar = AnchoredSizeBar(ax.transAxes, scalebar_length/(2*pc_width), f'{scalebar_length} pc', 'lower left', pad=0.1, color='white', frameon=False, size_vertical=.01)
ax.add_artist(scalebar)

### Colorbar
vmin_value = 2.5e2  # Minimum value for the color bar 
vmax_value = 5e4    # Maximum value for the color bar
axcb = plt.subplot(gs0[num_snapshots])
cb = colorbar.ColorbarBase(axcb, norm=LogNorm(vmin=vmin_value, vmax=vmax_value))
cb.set_label('$\Sigma$ (M$_{\odot}$/pc$^2$)')

plt.show()

### Your Turn:

***Make your own picture of a star cluster like on the example above!*** 

Try changing some of the values such as stepn, kpc_width, or cluster select.


***Expand cells below to show a solution***

In [ ]:
#establishing some units 
b_parsec    = 4
b_kpc       = b_parsec / 1000.
ncut_min    = 5
softening   = 4. #res 7100

# Initialize dictionaries to store properties
gas_star_properties_data_solution = {}
part_solution = {}

snaps_solution = [696]


for snapshot in snaps_solution:
    # Read the snapshot data for the current snapshot
    part_solution[snapshot] = gizmo.io.Read.read_snapshots(['star', 'gas'],'index', snapshot, simulation_directory=simdir, assign_hosts_rotation=True,assign_hosts=True)
    # Get star properties for the current snapshot
    gas_star_properties_data_solution[snapshot] = star_cluster_imaging.get_gas_star_properties(part_solution[snapshot])

In [ ]:
# Process the snapshots to get initial cluster information
cluster_select_solution = 0 #change this to any cluster as long as it is within range of the 'number of groups' value after running first_snapshot_data, 0 being the first cluster 
first_snapshot_data_solution = star_cluster_imaging.process_first_snapshot(gas_star_properties_data_solution[snaps_solution[0]],age_cut_Max_Myr=2, age_cut_Min_Myr=1, cluster_select = cluster_select_solution )

# Initialize variables
pc_width = 40 # this changes the zoom of the image. Based on this value the total width of the picture will be 2*pc_width
kpc_width = pc_width / 1e3
num_snapshots = len(snaps_solution)

plot_size = 5
fig = plt.figure(figsize=(num_snapshots*plot_size, plot_size))
gs0_solution = gridspec.GridSpec(1, num_snapshots+1, width_ratios= [1] * num_snapshots + [0.1])

gs_solution = gas_star_properties_data_solution[snapshot]
   
s_loc = star_cluster_imaging.preserve_ids(
    first_snapshot_data_solution['ids_cluster'],
    first_snapshot_data_solution['id_generation_cluster'],
    first_snapshot_data_solution['id_child_cluster'], 
    gs_solution['ids'], 
    gs_solution['id_generation'],
    gs_solution['id_child'])

vmin_value = 2.5e2  # Minimum value for the color bar 
vmax_value = 5e4    # Maximum value for the color bar

cluster_info = star_cluster_imaging.find_new_snap_info(s_loc, gs_solution)

mass_cluster = cluster_info['mass_cluster']
x_cluster = cluster_info['x_cluster']
y_cluster = cluster_info['y_cluster']
z_cluster = cluster_info['z_cluster']
    

xcm1 = first_snapshot_data_solution['sxcm'][cluster_select]
ycm1 = first_snapshot_data_solution['sycm'][cluster_select]
zcm1 = first_snapshot_data_solution['szcm'][cluster_select]


#creating filter that will only select the young stars that are visible in the picture
g_ind, s_ind = star_cluster_imaging.get_gas_star_indices(xcm1, ycm1, zcm1, gs_solution, pc_width=pc_width)
    
gas_star_coordinates = star_cluster_imaging.gas_star_coordinates(gs_solution, g_ind, x_cluster, y_cluster, z_cluster)

#Gaussian Smoothing for gas particles
vel_array_xy, res_xy, extent_xy = star_cluster_imaging.compute_velocity_array_xy(xcm1, ycm1, zcm1, gas_star_coordinates, kernel_file_path, pc_width=pc_width, stepn=13) # change the value of stepn to change the gas resolution
vel_array_xy = np.where(vel_array_xy < vmin_value, vmin_value, vel_array_xy) #This line is important for filling in any empty spaces in the gaussian smoothing. without it you end up with ugly white spaces when there is not gas in a region. 
ax = plt.subplot(gs0_solution[0])
ax.get_xaxis().set_visible(False) #remove x-axis labels
ax.get_yaxis().set_visible(False) #remove y-axis labels
ax.set_xlim(xcm1 - kpc_width, xcm1 + kpc_width) #keeping the image within kpc_width
ax.set_ylim(ycm1 - kpc_width, ycm1 + kpc_width) #keeping the image within kpc_width

#PLOT GAS 
ax.imshow(vel_array_xy.T / ((res_xy * 1000.) ** 2), interpolation='gaussian', norm=LogNorm(vmin=vmin_value, vmax=vmax_value), extent=extent_xy, origin='lower')

#PLOT STARS 
ax.scatter(x_cluster, y_cluster, s=15, marker='*', edgecolors='black', facecolors='yellow', linewidth=0.5) #This plots the stars, you can edit the colors and sizes to your liking

### Plain text
ax.text(0.01 ,0.99,'X Y', ha = 'left', va ='top', transform = ax.transAxes, c = 'white')
ax.text(0.99 ,0.01, snaps_solution[0], ha = 'right', va ='bottom', transform = ax.transAxes, c = 'white')

### Making Time Labels
snapshot_times = simdir + '/snapshot_times.txt' # Reading in the snapshot times from a text file to create titles for the images
snapnumber = snaps[0]  
# Reading the snapshot times and associated data from the text file
snaptime_data = np.genfromtxt(snapshot_times, usecols=(0, 3), skip_header=4, dtype=float)
snaps1 = np.array(snaptime_data[:, 0]) # Extracting the snapshot numbers (first column) from the data
times = np.array(snaptime_data[:, 1]) # Extracting the corresponding times (fourth column) from the data
snaptime = times[np.where(snaps1 == snapnumber)][0]# Finding the time associated with the current snapshot number
titles.append(f'XY VIEW {snaptime:.3f} Gyr')# Creating a title string that includes the snapshot time for the XY view
title = f'{snaptime:.3f}' + ' Gyr'# Creating a general title string for the snapshot time, formatted to 3 decimal places
ax.text(0.99 ,0.99,title, ha = 'right', va ='top', transform = ax.transAxes, c = 'white') #positioning and adding time label 


### Scalebar
scalebar_length = 20 #pc    
scalebar = AnchoredSizeBar(ax.transAxes, scalebar_length/(2*pc_width), f'{scalebar_length} pc', 'lower left', pad=0.1, color='white', frameon=False, size_vertical=.01)
ax.add_artist(scalebar)

### Colorbar
vmin_value = 2.5e2  # Minimum value for the color bar 
vmax_value = 5e4    # Maximum value for the color bar
axcb = plt.subplot(gs0_solution[num_snapshots])
cb = colorbar.ColorbarBase(axcb, norm=LogNorm(vmin=vmin_value, vmax=vmax_value))
cb.set_label('$\Sigma$ (M$_{\odot}$/pc$^2$)')

plt.show()